## Agentic AI Analytics Bot

Features:
- 6-stage analytics pipeline: interpret → classify → plan → execute → narrate → guardrails
- LLM-based mode selection (Retrieve / Explore / Reason)
- Dual mode control: auto (LLM picks per interaction) or manual (slash commands lock mode)
- Slash commands: /retrieve /explore /reason (lock) · /auto (unlock)
- Three-tier model routing: Haiku/8B-instant (fast), Sonnet/Llama-70B (default), Opus 4.7 / GPT-OSS-120B (Reason narration)
- Rolling session summary: bounded-context follow-up resolution across long sessions
- Automatic schema-aware query generation (DuckDB)
- DOC_LOOKUP: answers metadata questions directly from data model documentation
- guard_sql: AST-level SELECT-only enforcement via sqlglot (blocks DDL, DML, system tables)
- SQL error retry: targeted per-query fix prompt (SQL_RETRY_PROMPT), not full re-plan
- verify_groundedness: regex number/entity extraction vs. query results (no LLM call)
- check_compliance: tier-0 LLM scan for 5 violation categories (approximation, evaluative, inference, unit drift, unsupported claims)
- Hard-block: narrative replaced with issue code-name on numeric validation failure; compliance check is log-only
- Injection-based guardrail eval harness: 18 cases across SQL / narrative / prompt injection stages
- Cross-model grading: assessor_llm_fn param in run_eval() and run_guardrail_eval()
- Codebase version management / git integration
- Centralized prompt management (prompts.py)

Validation: 50/50 standard suite (Mode 0) · guardrail injection suite pending full run

## Quick Start
1. Run Blocks 1–5 in order (one-time setup)
2. **Chat mode:** Run Block 6 — type questions, type `quit` to exit
3. **Validation:** In Block 7, uncomment `results = run_eval(...)` and run
4. **Ad-hoc SQL:** After Block 3, use `run_query("SELECT ...")` in any cell
5. **Iterate on prompts:** Edit `prompts.py`, then rerun Block 5, 6 or 7 (modules auto-reload)

In [ ]:
# ============================================================
# Block 1: Environment setup
# ============================================================

# ── Project structure ───────────────────────────────────────
PROJECT = "Analytics Agent"
VERSION = "1.6.1"

REPO    = "https://github.com/ehasin/Analytics-Agent.git"
BRANCH  = "feature/guardrails"   # ← change per branch you want to test
DEST    = "/content/analytics-agent"

CODE_MODULES_SUBDIR = "code_modules"
DATA_SCHEMA_SUBDIR  = ""
SCHEMA_FILE_NAME    = "data_model.json"

# ── Optional persistent logging ─────────────────────────────
PERSISTENT_LOGGING = "NO"   # ← change to YES for GDrive output (default = NO)
LOGS_LOCATION = "Projects/Analytics Agent Logs"  # path relative to MyDrive

# ── Log file prefix ──────────────────────────────────────────
# Produces filenames like: Analytics_Agent_1.6.1_feature-guardrails_chat_log_...
LOG_PREFIX = f"{PROJECT.replace(' ', '_')}_{VERSION}_{BRANCH.replace('/', '-')}_"

# ── API Secret names ────────────────────────────────────────
# must match what you set in Colab Secrets
API_SECRET_NAMES = {
    "claude": "Claude",
    "groq":   "Groq",
    "gemini": "PersonalAPIKey",
}

print("All set")

In [ ]:
# ============================================================
# Block 2: Master initialization
# ============================================================

import importlib, subprocess, sys, shutil, os
from datetime import datetime
from pathlib import Path

# ── Install packages ────────────────────────────────────────
INSTALL_PACKAGES = ["anthropic", "groq", "duckdb", "pydrive2"]

def ensure_packages(packages):
    for pkg in packages:
        try:
            importlib.import_module(pkg)
        except ImportError:
            print(f"Installing: {pkg}")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure_packages(INSTALL_PACKAGES)
print("Packages OK")

# ── Clone / pull repo ────────────────────────────────────────
if not os.path.exists(DEST):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, DEST], check=True)
else:
    subprocess.run(["git", "-C", DEST, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", DEST, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", DEST, "pull"], check=True)

print("Branch:", subprocess.check_output(["git", "-C", DEST, "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip())
print("Commit:", subprocess.check_output(["git", "-C", DEST, "log", "--oneline", "-1"]).decode().strip())
print("Repo OK")

# ── Register code_modules on sys.path ────────────────────────
CODE_MODULES_DIR = Path(DEST) / CODE_MODULES_SUBDIR
DATA_SCHEMA_DIR  = Path(DEST) / DATA_SCHEMA_SUBDIR
SCHEMA_PATH      = DATA_SCHEMA_DIR / SCHEMA_FILE_NAME

assert CODE_MODULES_DIR.exists(), f"code_modules not found at {CODE_MODULES_DIR}"

for d in [CODE_MODULES_DIR] + [CODE_MODULES_DIR / sub for sub in ["_data", "_skills", "_agent"]]:
    shutil.rmtree(str(d / "__pycache__"), ignore_errors=True)

for p in [DEST, str(CODE_MODULES_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Code modules: {CODE_MODULES_DIR}")
print(f"Schema:       {SCHEMA_PATH}")
print("Modules OK")

# ── Persistent logging ────────────────────────────────────────
if PERSISTENT_LOGGING == "YES":
    def ensure_drive_mounted(mount_point="/content/drive"):
        mount_point = Path(mount_point)
        if (mount_point / "MyDrive").exists():
            return
        from google.colab import drive
        drive.mount(str(mount_point))
    ensure_drive_mounted()
    LOG_RUN_DIR = f"/content/drive/MyDrive/{LOGS_LOCATION}"
    print(f"GDrive mounted for persistent logging → {LOG_RUN_DIR}")
else:
    LOG_RUN_DIR = "/content/logs"
    print(f"Local logging → {LOG_RUN_DIR}")

# ── Reload helper ────────────────────────────────────────────
def reload_code_modules():
    """Reload all project modules from repo after editing source files."""
    import types
    prefix = str(CODE_MODULES_DIR)
    reloaded = []
    for name, mod in list(sys.modules.items()):
        if isinstance(mod, types.ModuleType) and hasattr(mod, "__file__") and mod.__file__ and mod.__file__.startswith(prefix):
            importlib.reload(mod)
            reloaded.append(name)
    if reloaded:
        print(f"Reloaded {len(reloaded)} modules")

In [7]:
# ============================================================
# Block 3: Backend config
# ============================================================

from _skills import llm_backends
reload_code_modules()

from _skills.llm_backends import init_backends, call_llm, MODEL_MAP

BACKEND = "claude"                       # "claude", "groq", or "gemini"
BACKENDS_TO_INIT = [BACKEND]             # init only what you need

result = init_backends(BACKENDS_TO_INIT, secret_names=API_SECRET_NAMES, test=True)
llm_clients = result["clients"]

for name, reply in result["pings"].items():
    print(f"  {name:<7} ✓  {reply}")

def llm(prompt, tier=1):
    return call_llm(prompt, backend=BACKEND, clients=llm_clients, model=tier)


Reloaded 2 modules
  claude  ✓  "Hi there, friend!" 👋


In [8]:
# ============================================================
# Block 4: Data — schema, datasets, DuckDB
# ============================================================

from _data import olist_schema_and_datasets
from _skills import duckdb_utils
reload_code_modules()

from _data.olist_schema_and_datasets import init_all
from _skills.duckdb_utils import run_query as duckdb_run_query, validate_tables

result = init_all(schema_path=SCHEMA_PATH, validate_fn=validate_tables)

data_model    = result["data_model"]
SCHEMA        = result["SCHEMA"]
tables_meta   = result["tables_meta"]
datasets      = result["datasets"]
duckdb_tables = result["duckdb_tables"]
tables        = duckdb_tables  # alias for agent functions

# ── Convenience: ad-hoc SQL from notebook ───────────────────
def run_query(sql):
    return duckdb_run_query(sql=sql, tables=tables)

Reloaded 5 modules
Schema loaded: 7 tables, 6,450 chars

  orders           99,441 rows  cols=['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
  products         32,951 rows  cols=['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
  reviews          99,224 rows  cols=['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
  customers        99,441 rows  cols=['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
  order_items     112,650 rows  cols=['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
  payments        10

In [ ]:
# ============================================================
# Block 5: Agent + Chat setup
# ============================================================

from _agent import (
    analyst_agent as _aa_mod,
    chat as _chat_mod,
    prompts as _prompts_mod,
    conversation_processor as _cp_mod,
)
from _skills import session_logger, eval_runner
reload_code_modules()

from _agent.analyst_agent import analyst_agent as _analyst_agent, MODE_NAMES
from _agent.chat import create_chat_fn
from _skills.duckdb_utils import execute_queries
from _skills.eval_runner import run_eval as _run_eval, run_guardrail_eval as _run_guardrail_eval

def agent(question, mode=0, user_context=None, execute_fn=None):
    """Agent wrapper. execute_fn is overridable so run_guardrail_eval
    can inject a custom executor without replacing the whole agent."""
    return _analyst_agent(
        question=question, schema=SCHEMA, tables=tables,
        llm_fn=llm, execute_fn=execute_fn or execute_queries,
        mode=mode, user_context=user_context,
    )

chat = create_chat_fn(
    agent_fn=agent,
    llm_fn=llm,
    logs_dir=LOG_RUN_DIR,
    backend=BACKEND,
    model_name=MODEL_MAP[BACKEND][1],
    log_prefix=LOG_PREFIX,
)

# ── Bound eval wrappers (logs_dir + log_prefix baked in) ────
# Use these in Block 7 — no need to repeat log settings per call.

def run_eval(test_sets, mode=0, assessor_llm_fn=None):
    """Standard eval: test_sets is {name: [cases]} with string validators."""
    return _run_eval(
        agent, llm, test_sets,
        logs_dir=LOG_RUN_DIR, log_prefix=LOG_PREFIX,
        mode=mode, assessor_llm_fn=assessor_llm_fn,
    )

def run_guardrail_eval(test_cases, mode=0, assessor_llm_fn=None):
    """Guardrail injection eval: test_cases is a flat list with result-dict validators."""
    return _run_guardrail_eval(
        agent, execute_queries, llm, test_cases,
        logs_dir=LOG_RUN_DIR, log_prefix=LOG_PREFIX,
        mode=mode, assessor_llm_fn=assessor_llm_fn,
        schema_context=SCHEMA,
    )

print("Agent is ready")

In [12]:
# ============================================================
# Block 6: Interactive Chat / Analytics Agent
# ============================================================

chat()

Session log: /content/Logs/chat_log_2026_05_07_10_18.md
Analytics Agent [claude]  •  Retrieve mode (auto)
Commands: /retrieve /explore /reason (lock mode) · /auto (unlock) · quit

You: say hi
  → Interpreting...
  → Interaction type: standalone
  → Mode: Retrieve
  → Planning and executing queries...


**Bot:** Can't answer based on the available data. (The user said "hi" — this is a greeting, not a data question. It is unanswerable in principle as an analytical query (category c under cant_answer). No data, metric, or dimension is implied.

*Hello! How can I help you today? Feel free to ask any question about the Olist dataset — order trends, seller performance, delivery KPIs, customer satisfaction, revenue analysis, and more.*)

  → Updating summary...
⏱  8.01s total

You: quit
Session ended.


In [ ]:
# ============================================================
# Block 7: Validation (optional)
# ============================================================

from _agent import prompts as _prompts_mod, analyst_agent as _aa_mod
from _skills import eval_runner
from _data import olist_test_cases, guardrail_test_cases
reload_code_modules()

from _data.olist_test_cases import discovery, easy, hard, misleading, multistage, realistic
from _data.guardrail_test_cases import all_guardrail_cases

from IPython.display import display, Markdown, HTML

# ── Custom one-off test questions ───────────────────────────

## Question (activate when needed)
q = "How many customers do we have?"

## Single-mode (activate when needed)
# r = agent(q, mode=0)
# display(Markdown(f"**Q:** {q}\n\n{r['narrative']}"))

## Compare modes (activate when needed)
# for m in [0, 1, 2]:
#     r = agent(q, mode=m)
#     display(Markdown(f"**Mode {m}:** {r['narrative']}"))


# ── Standard eval (string validators) ───────────────────────
# run_eval(test_sets, mode=0, assessor_llm_fn=None)
# test_sets: {name: [cases]}  — validators receive answer/narrative text

## Pick which sets to run — comment/uncomment as needed
validation = {
    "Metadata / Descriptive": discovery,
    "Easy": easy,
    "Hard": hard,
    "Misleading": misleading,
    "Multi-stage": multistage,
    "Realistic": realistic,
}

results = run_eval(validation, mode=0)


# ── Guardrail injection eval (result-dict validators) ────────
# run_guardrail_eval(test_cases, mode=0, assessor_llm_fn=None)
# test_cases: flat list — validators receive the full result dict
#             (stage_trace, grounding, compliance, retry, etc.)
# SQL injection, narrative injection, and prompt injection cases
# MUST use run_guardrail_eval — they are not compatible with run_eval.

## Uncomment to run:
# g_results = run_guardrail_eval(all_guardrail_cases, mode=0)

## Validation Results

**50 / 50 questions passed** across all test categories (Mode 0 — Retrieve):

| Test Set | Passed | Total |
|---|---|---|
| Metadata / Descriptive | 6 | 6 |
| Easy | 10 | 10 |
| Hard | 10 | 10 |
| Misleading | 10 | 10 |
| Multi-stage | 5 | 5 |
| Realistic | 9 | 9 |
| **Total** | **50** | **50** |

Test categories cover: schema/metadata questions, simple aggregations, complex multi-join queries, edge cases designed to trigger misclassification, multi-step follow-up chains, and realistic business analyst questions. LLM-assessed reconciliation applied to all results.